In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('Carcinogenicity.csv')
df.shape

In [ ]:
df = df.dropna()
df.shape

In [ ]:
!pwd

In [ ]:
from rdkit import Chem
from standardiser import standardise
import logging
from rdkit.Chem import Descriptors
import rdkit

In [ ]:
for i in df.index:
    try:
        smi = df.loc[i,'SMILES']
        mol = Chem.MolFromSmiles(smi)
        mol = Chem.AddHs(mol)
        parent = standardise.run(mol)  # 使用standatdise清洗分子
        mol_ok_smi = Chem.MolToSmiles(parent)
        df.loc[i,'SMILES'] = mol_ok_smi
        print(i,'done')
    except standardise.StandardiseException as e:
              logging.warning(e.message)

In [ ]:
df.shape

In [ ]:
df.drop_duplicates(keep='first',inplace=True)

In [ ]:
df.shape

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
import pandas as pd
import warnings

# Set RDKit log level to ERROR globally to suppress WARNING messages
lg = RDLogger.logger()
lg.setLevel(RDLogger.ERROR)

def calc_all_descriptors(smiles):
    """
    Calculate 6 molecular descriptors from SMILES.
    If SMILES is invalid or contains isolated hydrogen atoms, returns [None]*6 and issues a warning.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [None] * 6

    # Detect isolated hydrogen atoms (atomic number 1 with degree 0)
    isolated_h = [atom for atom in mol.GetAtoms() if atom.GetAtomicNum() == 1 and atom.GetDegree() == 0]
    if isolated_h:
        warnings.warn(f"SMILES '{smiles}' contains isolated hydrogen atoms, descriptors will be set to None")
        return [None] * 6

    # Calculate descriptors
    return [
        Descriptors.MolWt(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.MolLogP(mol),
        Descriptors.MolMR(mol)
    ]

# Assume df already exists and contains the 'SMILES' column
# Apply the function to obtain a list of lists
results = df['SMILES'].apply(calc_all_descriptors)

# Split the results into multiple columns (more efficient)
df[['MolWt', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'LogP', 'MolMR']] = pd.DataFrame(
    results.tolist(), index=df.index
)

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.to_csv('data.csv',index=None)

In [ ]:
import numpy as np
import pandas as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import MACCSkeys
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import roc_auc_score, confusion_matrix
import math

In [ ]:
# Load dataset
df = pd.read_csv('data.csv')